***Before running this notebook, make sure you have kaggle.json file in your home directory***

In [ ]:
# Import libraries
import sys
sys.path.insert(0, "..")

import pandas as                          pd
import numpy as                           np
import matplotlib.pyplot as               plt
from sklearn.preprocessing import         LabelEncoder

from src.preprocessing import (
    fill_missing_values,
    sync_data_column,
    get_categorical_features,
    get_numerical_features,
    extract_date,
    skewness_analyze,
    fix_skewness,
    method_preprocessing,
    align_columns,
)

# Load and Visualize 

In [2]:
!kaggle competitions download -c store-sales-time-series-forecasting

zsh:1: command not found: kaggle


In [3]:
import zipfile 
import os 

archive = zipfile.ZipFile('store-sales-time-series-forecasting.zip')

for file in archive.namelist():
    archive.extract(file, '../kaggle-dataset')

FileNotFoundError: [Errno 2] No such file or directory: 'store-sales-time-series-forecasting.zip'

In [ ]:
os.remove("store-sales-time-series-forecasting.zip")

In [ ]:
# Load data

train_dataset = pd.read_csv('../kaggle-dataset/train.csv') 
test_data = pd.read_csv('../kaggle-dataset/test.csv')

# Sub data
oil_data = pd.read_csv('../kaggle-dataset/oil.csv')
holiday_events = pd.read_csv('../kaggle-dataset/holidays_events.csv') 
stores_csv = pd.read_csv('../kaggle-dataset/stores.csv')
# Note: We are not using transactions.csv since it doesn't contain data for test.csv

train_dataset[:3]

In [ ]:
print(f"Min sales: {min(train_dataset['sales'])}")
print(f"Max sales: {max(train_dataset['sales'])}")
print(f"Mean sales: {np.mean(train_dataset['sales'])}")
print(f"Train shape: {train_dataset.shape}")
print(f"Test shape: {test_data.shape}")
print(f"There are {len(train_dataset['sales'][train_dataset['sales'] > 0].unique())} unique values of sales in train data")
print(f"There are {len(train_dataset['sales'][train_dataset['sales'] == 0])} rows with 0 sales in train data")
print(f"There are {len(train_dataset['family'].unique())} unique families")

# Dtypes 
train_dataset.dtypes

# Preprocess data

In [ ]:
# Check is there any missing values
print(f"Missing values for oil data: {oil_data.isna().sum().sum()}")
print(f"Missing values for holiday data: {holiday_events.isna().sum().sum()}")
print(f"Missing values for stores data: {stores_csv.isna().sum().sum()}")

In [ ]:
# fill_missing_values() is imported from src.preprocessing
# It fills gaps by interpolating between adjacent known values,
# using the mean change as fallback for edge positions.

## **Sub Data: Row Inserting and Processing** 

Let's find all the **"date"** rows that our **oil_price** dataset doesn't have compared with our main datasets. It will help in future with filling our main datasets

In [ ]:
# sync_data_column() is imported from src.preprocessing
# It aligns dates between auxiliary datasets and train/test data.

### Preprocess oil.csv

In [ ]:
%%time
# Sync oil data with train and test dataset
oil_data = sync_data_column(oil_data, train_dataset, test_data)
print(f"Missing values for oil data before: {oil_data.isna().sum().sum()}")

# Fill missing values
oil_data["oil_price"] = fill_missing_values(oil_data["dcoilwtico"])
oil_data.drop(columns=["dcoilwtico"], axis=1, inplace=True)

print(f"Missing values for oil data after: {oil_data.isna().sum().sum()}")

oil_data.head(10)

### Preprocess holidays data

## **Preprocessing Main Datasets**

In [ ]:
# get_categorical_features() and get_numerical_features() imported from src.preprocessing

In [ ]:
# (see src/preprocessing.py for get_numerical_features)

In [ ]:
%%time
train_dataset['date'] = pd.to_datetime(train_dataset['date']).astype('datetime64[ns]')
test_data['date'] = pd.to_datetime(test_data['date']).astype('datetime64[ns]')
oil_data['date'] = pd.to_datetime(oil_data['date']).astype('datetime64[ns]')
holiday_events['date'] = pd.to_datetime(holiday_events['date']).astype('datetime64[ns]')

train_dataset = train_dataset.merge(oil_data, on="date", how="left")
train_dataset = train_dataset.merge(holiday_events, on="date", how="left")
train_dataset = train_dataset.merge(stores_csv, on="store_nbr", how="left")

test_data = test_data.merge(oil_data, on="date", how="left")
test_data = test_data.merge(holiday_events, on="date", how="left")
test_data = test_data.merge(stores_csv, on="store_nbr", how="left")


# Label Encode Categorical Features
categorical_features = get_categorical_features(train_dataset)

encoder = LabelEncoder()
train_dataset[categorical_features] = train_dataset[categorical_features].apply(encoder.fit_transform)
test_data[categorical_features] = test_data[categorical_features].apply(encoder.fit_transform)

print(f"Missing values for train dataset: {train_dataset.isna().sum().sum()}")
print(f"Missing values for test dataset: {test_data.isna().sum().sum()}")
train_dataset.head(10)


### Additional Note: Wages in the public sector are paid every two weeks on the 15th and on the last day of the month. Supermarket sales could be affected by this.

In [ ]:
# Wages in Ecuador are paid on the 15th and last day of the month.
# Supermarket sales are affected by this — add a WageDay indicator.
from src.preprocessing import add_wage_day

train_dataset = add_wage_day(train_dataset)
test_data = add_wage_day(test_data)

train_dataset["WageDay"]

## Deal with skewness

In [ ]:
# skewness_analyze() and fix_skewness() imported from src.preprocessing

def compare_skewed_preprocessing(df, skewed_columns, preprocessed=False):
    """Visualize skewness before and after log transformation."""
    n_features = len(skewed_columns)
    fig, axs = plt.subplots(n_features, 2, figsize=(20, n_features * 5))

    for i, col in enumerate(df[skewed_columns]):
        axs[i, 0].hist(df[col], bins=30, edgecolor='k')
        axs[i, 0].set_title(f'Histogram of {col}')
        axs[i, 0].set_xlabel('Value')
        axs[i, 0].set_ylabel('Frequency')

        if preprocessed:
            df_fixed = fix_skewness(df.copy(), skewed_columns)
            axs[i, 1].hist(df_fixed[col], bins=30, edgecolor='k')
            axs[i, 1].set_title(f'Log-Transformed Histogram of {col}')
            axs[i, 1].set_xlabel('Value')
            axs[i, 1].set_ylabel('Frequency')

    plt.tight_layout()
    plt.show()

skewed_columns = skewness_analyze(train_dataset)
compare_skewed_preprocessing(train_dataset, skewed_columns, preprocessed=True)

## Finish preprocessing

In [ ]:
# extract_date() is imported from src.preprocessing
# Extracts: year, month, day, day_of_week, week_of_year, quarter, is_weekend, is_leap_year, is_month_end, is_month_start

In [ ]:
def df_dtypes(df):
      pd.set_option('display.max_colwidth', None)
  
      df_dtypes = df.columns.groupby(df.dtypes)
      print(list(df_dtypes.keys()))
      df_dtypes = pd.DataFrame({
          'dtype':     list(df_dtypes.keys()),
          '# columns': [len(df_dtypes[key])  for key in df_dtypes.keys()],
          'columns':   [list(df_dtypes[key]) for key in df_dtypes.keys()],
      })
      df_dtypes = df_dtypes.style.applymap(lambda x:'text-align: left', subset=['columns'])
      return df_dtypes

df_dtypes(train_dataset)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

def one_hot_encoding(df):
    cat_features = get_categorical_features(df)
    print(f'Categorical features: {cat_features}')

    encoder = OneHotEncoder(drop='first', sparse_output=False)
    one_hot = encoder.fit_transform(df[cat_features])
    hot_df = pd.DataFrame(one_hot, columns=encoder.get_feature_names_out(cat_features), index=df.index)
    print(hot_df.shape)
    df = df.drop(columns=cat_features)
    df = pd.concat([df, hot_df], axis=1)
    return df

In [ ]:
# Finish preprocessing
train_dataset.drop(columns=['id'], inplace=True)
test_data.drop(columns=['id'], inplace=True) 

extract_date(train_dataset, 'date') # type: ignore
extract_date(test_data, 'date') # type: ignore

# Replace true and false for 1 and 0 in transferred column
train_dataset['transferred'] = train_dataset['transferred'].astype(int)
train_dataset['WageDay'] = train_dataset['WageDay'].astype(int)
test_data['transferred'] = test_data['transferred'].astype(int)
test_data['WageDay'] = test_data['WageDay'].astype(int)

train_dataset = one_hot_encoding(train_dataset) # type: ignore
test_data = one_hot_encoding(test_data) # type: ignore

train_dataset.info()

In [ ]:
df_dtypes(train_dataset)

## Column alighnment with train and test sets 

In [ ]:
print(f"Number of columns in train dataset: {len(train_dataset.columns)}")
print(f"Number of columns in test dataset: {len(test_data.columns)}")

In [ ]:
# align_columns() imported from src.preprocessing
train_dataset, test_data = align_columns(train_dataset, test_data, fix_columns=True)
print(f"Number of columns in train dataset: {len(train_dataset.columns)}")
print(f"Number of columns in test dataset: {len(test_data.columns)}")

In [ ]:
train_dataset

### Split Dataset

In [ ]:
split = int(len(train_dataset) * 0.7)
train_dataset, validation_dataset = train_dataset[:split], train_dataset[split:]

len(train_dataset), len(validation_dataset)

Three ways to organize our data: 
- 1. Do nothing. Leave the data unbalanced, we are getting more samples, but the predictions are awful and not correct.
- 2. Delete all zero sales. Good way, but then there's no probability for zeros. 
- 3. Leave a small percent of all zeros in our dataset. So when we are going to predict, the model will also count the probabilities of data to be zero sales. 

In [ ]:
# How much zero sales in our dataset? # How much sales > 0 in our dataset?
len(train_dataset[train_dataset["sales"] == 0]), len(train_dataset[train_dataset["sales"] > 0])

In [ ]:
# method_preprocessing() imported from src.preprocessing
# Method 3: Keep 20% of zero-sales rows to preserve zero probability

In [ ]:
train_dataset = method_preprocessing(train_dataset, method=3)

# Length after undersampling
len(train_dataset[train_dataset["sales"] == 0]), len(train_dataset[train_dataset["sales"] > 0])

In [ ]:
## Are there are missing values? 
train_dataset.isnull().sum().sum()

# Save processed data 

** All the datasets weight ~2.7 Gb **

In [ ]:
def memory_usage(dfs): 
    def calculate_memory_usage(df):
        mem_usage_bytes = df.memory_usage(deep=True).sum()
        mem_usage_gb = mem_usage_bytes / (1024 ** 3)
        return mem_usage_gb

    
    if isinstance(dfs, dict):
        for name, df in dfs.items():
            mem_usage_gb = calculate_memory_usage(pd.DataFrame(df))
            print(f"{name} memory usage in gigabytes: {mem_usage_gb:.2f} GB")
    else:
        mem_usage_gb = calculate_memory_usage(dfs)
        print(f"Memory usage in gigabytes: {mem_usage_gb:.2f} GB")
        return {"gigabytes": mem_usage_gb}

    
# Accepts a dictionary of dataframe
memory_usage({
    "train_dataset": train_dataset,
    "validation_dataset": validation_dataset,
    "test_dataset": test_data})

In [ ]:
import os

train_file = "../preprocessed_dataset/train_dataset.csv"

# Save data
if not os.path.exists(train_file): 
    os.makedirs("../preprocessed_dataset", exist_ok=True)
    train_dataset.to_csv('../preprocessed_dataset/train_dataset.csv', index=False)
    # No sampling for validation_dataset for correct evaluation.
    validation_dataset.to_csv('../preprocessed_dataset/validation_dataset.csv', index=False)
    test_data.to_csv('../preprocessed_dataset/test_dataset.csv', index=False)
    print("Data was successfully saved.")
else: 
    print(f'File {train_file} already exists and will not be overwritten.')